In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("/content/final_df.csv")

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report


In [4]:
# Skill columns
skill_columns = [
    'Skills - Python',
    'Skills - SQL',
    'Skills - ML',
    'Skills-DeepLearning',
    'skills_Cloud'
]

# Other useful columns
other_features = [
    'Years_Experience',
    'Education  Level',
    'Exp. Level',
    'Remote Type'
]

# Final features
features = skill_columns + other_features

# Target column
target = 'JobTitle'

X = df[features]
y = df[target]


In [5]:
categorical_cols = [
    'Education  Level',
    'Exp. Level',
    'Remote Type'
]

numerical_cols = [
    'Years_Experience'
]


In [6]:
# OneHotEncode categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_cols
        )
    ],
    remainder='passthrough'
)


In [7]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),

    ('model', RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    ))
])


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [9]:
# trainig model
pipeline.fit(X_train, y_train)

# prediction
y_pred = pipeline.predict(X_test)

In [10]:
print("Accuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy Score:
0.17605985037406482

Classification Report:
                           precision    recall  f1-score   support

              AI Engineer       0.14      0.10      0.12       336
         Business Analyst       0.19      0.33      0.24       343
             Data Analyst       0.21      0.07      0.10       325
            Data Engineer       0.18      0.12      0.14       325
           Data Scientist       0.19      0.18      0.19       332
Machine Learning Engineer       0.16      0.26      0.20       336
                  Unknown       0.00      0.00      0.00         8

                 accuracy                           0.18      2005
                macro avg       0.15      0.15      0.14      2005
             weighted avg       0.18      0.18      0.16      2005



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
df['JobTitle'].unique()

array(['Data Analyst', 'Data Scientist', 'Machine Learning Engineer',
       'AI Engineer', 'Business Analyst', 'Data Engineer', 'Unknown'],
      dtype=object)

In [13]:
df['JobTitle_Original'] = df['JobTitle']

In [14]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['JobTitle'] = le.fit_transform(df['JobTitle'])

In [16]:
import joblib

In [17]:
joblib.dump(le, "jobtitle_label_encoder.pkl")

['jobtitle_label_encoder.pkl']

In [18]:
skill_columns = [
    'Skills - Python',
    'Skills - SQL',
    'Skills - ML',
    'Skills-DeepLearning',
    'skills_Cloud'
]

role_profiles = df.groupby('JobTitle_Original')[
    skill_columns
].mean()

print(role_profiles.head())

                   Skills - Python  Skills - SQL  Skills - ML  \
JobTitle_Original                                               
AI Engineer               0.427635      0.560453     0.567004   
Business Analyst          0.421237      0.564761     0.596849   
Data Analyst              0.434809      0.586716     0.556581   
Data Engineer             0.406038      0.561922     0.573629   
Data Scientist            0.420831      0.573751     0.571945   

                   Skills-DeepLearning  skills_Cloud  
JobTitle_Original                                     
AI Engineer                   0.444908      0.571769  
Business Analyst              0.425904      0.571762  
Data Analyst                  0.420049      0.563346  
Data Engineer                 0.426371      0.595194  
Data Scientist                0.414208      0.558098  


In [19]:
import joblib

In [20]:
joblib.dump(role_profiles, "role_job_profiles.pkl")

['role_job_profiles.pkl']

In [ ]:
role_profiles = joblib.load("role_profiles.pkl")

In [21]:
def analyze_skill_gap(candidate_skills, target_role):

    # If target role is encoded integer
    if isinstance(target_role, int):

        target_role = le.inverse_transform([target_role])[0]

    role_data = role_profiles.loc[target_role]

    missing_skills = []

    matched_skills = 0

    total_required = 0

    for skill, importance in role_data.items():

        if importance >= 0.5:

            total_required += 1

            if candidate_skills.get(skill, 0) == 1:
                matched_skills += 1

            else:
                missing_skills.append(skill)

    readiness_score = (
        matched_skills / total_required
    ) * 100

    return {
        "target_role": target_role,
        "missing_skills": missing_skills,
        "readiness_score": round(readiness_score, 2)
    }

In [22]:
df['JobTitle'].unique()

array([2, 4, 5, 0, 1, 3, 6])

In [23]:
candidate = {
    'Skills - Python': 1,
    'Skills - SQL': 0,
    'Skills - ML': 1,
    'Skills-DeepLearning': 0,
    'skills_Cloud': 1
}


result = analyze_skill_gap(
    candidate_skills=candidate,
    target_role=4
)
print(result)

{'target_role': 'Data Scientist', 'missing_skills': ['Skills - SQL'], 'readiness_score': 66.67}
